# Z.AI GLM 5.3 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

GLM 5.3 is the newest generation of Z.AI's GLM models. Unlike the models in
[`01-glm-family`](01-glm-family.ipynb), it is on `bedrock-runtime` only, behind an
inference profile, and it serves both Chat Completions and the Responses API there.

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| GLM-5.3 | — | `us.zai.glm-5.3` | Inference profile only, `global.zai.glm-5.3` where there is no `us.` profile |

IDs as of October 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

**Covered:** the inference profile, reasoning on Converse and its `reasoning_effort`
values, Chat Completions and the Responses API on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check, a translation from a `bedrock-mantle` model ID to the `bedrock-runtime` form
(`runtime_id_for()`), and the inference profile your Region carries
(`resolve_runtime_id()`).

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, resolve_runtime_id, runtime_id_for

REGION = "us-east-1"
GLM53 = "zai.glm-5.3"

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
# GLM 5.3 is not on bedrock-mantle, so this notebook builds no bedrock-mantle client.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Print the ID each endpoint takes rather than a yes/no: GLM 5.3 is profile-only on
# bedrock-runtime and absent from bedrock-mantle, so a bare "runtime" would invite the
# bare ID, which is refused (section 1). Resolve the runtime ID instead of reusing the string.
for model in (GLM53,):
    served = endpoints_for(model, REGION)
    mantle_id = model if served["mantle"] else "-"
    runtime_id = runtime_id_for(model, REGION) or "-"
    print(f"{model:19} mantle: {mantle_id:19} runtime: {runtime_id}")

zai.glm-5.3         mantle: -                   runtime: us.zai.glm-5.3


## 1. Calling GLM 5.3 on `bedrock-runtime`, and its reasoning

GLM 5.3 is on `bedrock-runtime` only. `bedrock-mantle` does not list it in `us-east-1`,
`us-west-2` or `eu-central-1`, and a call there returns 404 `The model 'zai.glm-5.3'
does not exist` (October 2026), so this notebook builds no `bedrock-mantle` client.

It is also `INFERENCE_PROFILE`-only: the bare ID is refused with `Invocation of model ID
zai.glm-5.3 with on-demand throughput isn’t supported`. Call the inference profile
instead, `us.zai.glm-5.3` in `us-east-1`. Some Regions carry only
`global.zai.glm-5.3`, so `resolve_runtime_id()` in the helper module returns the one that
exists in your Region rather than assuming a prefix.

GLM 5.3 reasons without being asked, and `reasoning_effort` selects how much. On
`us.zai.glm-5.3` in `us-east-1` in October 2026, Converse took `none`, `low`, `medium`,
`high` and `xhigh` on 12 of 12 calls each and `max` on 3 of 3, and refused `minimal`
with 400
`Unsupported value: 'reasoning_effort' does not support 'minimal' with this model.`,
on the `global.` profile as well as the `us.` one. A value the request schema does not
know at all, such as `banana`, fails earlier, in deserialization, and that error names a
wider set than the model takes, `minimal` among them, so read the values off a call
rather than off a parse error.

How much trace comes back varies enough between calls that a re-run prints a different
number, so read these as orders of magnitude and not as bounds. Measured on the prompt
in the cell below, 12 calls per setting, October 2026: with no `reasoning_effort` set
all 12 calls returned a trace of a few hundred characters, and `xhigh` was similar.
`none`, `low`, `medium` and `high` all returned less than that, despite the names.
`high` returned no `reasoningContent` block at all on 5 of 12 calls and at most 37
characters on the rest, `medium` no block on 3 of 12 and at most 41 characters
otherwise, `low` no block on 10 of 12, and `none` no block on 12 of 12. A short or
absent trace is normal here, not an error.

Setting `none` also costs accuracy on this prompt. It answered $0.05 on 3 of 12 calls,
where omitting `reasoning_effort` answered it on 12 of 12. The cell below therefore
omits the parameter rather than setting `none`, and shows the default reasoning the
paragraph above describes.

In [2]:
GLM53_ID = resolve_runtime_id(GLM53, REGION)  # us. or global., whichever this Region has
BALL = ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
        "How much does the ball cost? Answer with the amount only.")

response = runtime.converse(
    modelId=GLM53_ID,
    messages=[{"role": "user", "content": [{"text": BALL}]}],
    inferenceConfig={"maxTokens": 4000},
)  # no reasoning_effort: it reasons by default, and "high" returns little or no trace
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
trace = "".join(b["reasoningContent"].get("reasoningText", {}).get("text", "")
                for b in blocks if "reasoningContent" in b)
print("profile    :", GLM53_ID)
print("blocks     :", [next(iter(b)) for b in blocks])
print("answer     :", text.strip(), "| 0.05 present:", "0.05" in text)
print("reasoning  :", len(trace), "characters")
print("stopReason :", response["stopReason"])

profile    : us.zai.glm-5.3
blocks     : ['text', 'reasoningContent']
answer     : $0.05 | 0.05 present: True
reasoning  : 275 characters
stopReason : end_turn


## 2. Chat Completions and the Responses API

GLM 5.3 also serves Chat Completions and the Responses API on the OpenAI-compatible path
of `bedrock-runtime`. The earlier GLM models on that endpoint serve Chat Completions
there too, and refuse the Responses API with 400 `The model 'zai.glm-5' does not support
the '/openai/v1/responses' API`, measured for `zai.glm-5`, `zai.glm-4.7` and
`zai.glm-4.7-flash` in `us-east-1` in October 2026. The OpenAI SDK reaches them with a
short-term Bedrock API key, pointed at the `bedrock-runtime` base URL. On the Responses API
the reasoning arrives as a separate output item ahead of the message, which is why the
cell takes `output_text` rather than the first output item.

In [3]:
CAPITAL = "What is the capital of Australia? Answer in one word."

glm53 = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

chat = glm53.chat.completions.create(
    model=GLM53_ID,
    messages=[{"role": "user", "content": CAPITAL}],
    max_tokens=1000,
)
choice = chat.choices[0]
print("Chat Completions:", (choice.message.content or "").strip(),
      "| finish_reason:", choice.finish_reason)

# store=False: nothing here chains from this response, so do not leave it on the service.
reply = glm53.responses.create(
    model=GLM53_ID,
    input=CAPITAL,
    max_output_tokens=1000,
    store=False,
)
answer = reply.output_text.strip()
print("Responses API   :", answer, "| status:", reply.status)
print("Canberra from both:", "Canberra" in (choice.message.content or "") and "Canberra" in answer)

Chat Completions: Canberra | finish_reason: stop


Responses API   : Canberra | status: completed
Canberra from both: True


## Things that differ

What trips people up on GLM 5.3, and what to do about it. Behaviour observed in
October 2026.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API on `bedrock-runtime` | `zai.glm-5`, `zai.glm-4.7` and `zai.glm-4.7-flash` on `/openai/v1/responses` return 400 `The model '…' does not support the '/openai/v1/responses' API` | GLM 5.3 serves it (section 2). For the others, use Chat Completions on `/openai/v1`, or Converse. |
| GLM 5.3 on `bedrock-mantle` | 404 `The model 'zai.glm-5.3' does not exist` | Call it on `bedrock-runtime` (section 1). The other GLM models are on `bedrock-mantle` too ([`01-glm-family`](01-glm-family.ipynb)). |
| GLM 5.3 with its bare ID | 400 `Invocation of model ID zai.glm-5.3 with on-demand throughput isn’t supported` | Call the inference profile, `us.zai.glm-5.3` or `global.zai.glm-5.3`. `resolve_runtime_id()` picks the one your Region has. |
| Where the reasoning block sits | GLM-5 returns `reasoningContent` before the text block on Converse, GLM 5.3 after it | Read content blocks by type. Indexing `content[0]` gets the trace for one model and the answer for the other. |
| A `reasoning_effort` value the schema takes and the model does not | `minimal` returns 400 `Unsupported value: 'reasoning_effort' does not support 'minimal' with this model. Supported values are: 'none', 'low', 'medium', 'high', and 'xhigh'.` | Use the values measured in section 1. An unknown value fails in deserialization instead, and that error lists a wider set than GLM 5.3 accepts. |
| Reasoning you did not ask for | GLM 5.3 returns a `reasoningContent` block with no `reasoning_effort` set, on 12 of 12 calls | Reasoning is on by default, and a short or absent trace is not an error. Over 12 calls per setting in October 2026, `none` returned no block at all on 12 of 12 calls, `low` on 10 of 12 and `high` on 5 of 12, with at most 37 characters when `high` did return one. Section 1 has the rest. |

## Next

- The other GLM models, on both endpoints:
  [`06-zai-glm/01`](01-glm-family.ipynb)
- Inference profiles and Converse on `bedrock-runtime`:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)